# Testing detection.py

This notebook tests the detectors in `src/detection.py` to verify that
the rule-based flags produce reasonable counts before we add ML detectors.

## Setup

Load the features file once. Subsequent cells operate on `apc_features`
in memory.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
from src.detection import add_rule_flags

apc_features = pd.read_csv('../data/processed/apc_features.csv',
                            sep=';', encoding='latin-1')
print(f"Loaded apc_features: {apc_features.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded apc_features: (1598277, 39)


## Verification: how the Halland bounding box was chosen

The bounds in `flag_gps_outside_halland` were not picked arbitrarily.
They are based on the actual GPS distribution in the January 2025 data.
We exclude the (0, 0) error rows first, then compute the 1st and 99th
percentile of latitude and longitude — meaning where 99% of valid
positions fall. The box used in the code is slightly wider than this
range, to leave margin for legitimate trips at the region's edges
without flagging them as anomalies.

In [4]:
# Filter out the (0,0) GPS errors so they don't skew the percentiles
real_gps = apc_features[(apc_features['latitude']  != 0) &
                        (apc_features['longitude'] != 0)]

print(f"Rows with valid (non-zero) GPS: {len(real_gps):,}")
print()

print("Latitude:")
print(f"  observed range:   {real_gps['latitude'].min():.4f} - {real_gps['latitude'].max():.4f}")
print(f"  1st - 99th pct:   {real_gps['latitude'].quantile(0.01):.4f} - {real_gps['latitude'].quantile(0.99):.4f}")
print(f"  bounds used:      56.1 - 57.5")
print()
print("Longitude:")
print(f"  observed range:   {real_gps['longitude'].min():.4f} - {real_gps['longitude'].max():.4f}")
print(f"  1st - 99th pct:   {real_gps['longitude'].quantile(0.01):.4f} - {real_gps['longitude'].quantile(0.99):.4f}")
print(f"  bounds used:      12.0 - 13.7")
print()

# Show what would be flagged
out_of_box = real_gps[~(real_gps['latitude'].between(56.1, 57.5) &
                       real_gps['longitude'].between(12.0, 13.7))]
print(f"Rows outside the box (and not (0,0)): {len(out_of_box):,}")
print()
print("Sample:")
print(out_of_box[['vehicleCode', 'arrival', 'latitude', 'longitude']].head().to_string(index=False))

Rows with valid (non-zero) GPS: 1,579,878

Latitude:
  observed range:   0.0833 - 57.4255
  1st - 99th pct:   56.4665 - 57.3132
  bounds used:      56.1 - 57.5

Longitude:
  observed range:   11.8627 - 16.1451
  1st - 99th pct:   12.1288 - 13.2413
  bounds used:      12.0 - 13.7

Rows outside the box (and not (0,0)): 10

Sample:
 vehicleCode             arrival  latitude  longitude
        4237 02/01/2025 19:02:09 55.750600  16.128400
        4237 02/01/2025 19:03:03 55.740330  16.129860
        4237 02/01/2025 19:05:23 55.708510  16.125948
        4237 02/01/2025 19:06:45 55.692930  16.127388
        4237 02/01/2025 19:07:36 55.680668  16.131268


## Run rule-based detectors

Apply all four rule-based detectors to the features data and count flags
both at row level (GPS rules) and trip level (load rules).

In [5]:
flagged = add_rule_flags(apc_features)
print(f"After rules: {flagged.shape}")
print()

print("Row-level flags:")
print(f"  flag_gps_zero:            {flagged['flag_gps_zero'].sum():,}")
print(f"  flag_gps_outside_halland: {flagged['flag_gps_outside_halland'].sum():,}")
print()

trips = flagged.dropna(subset=['trip']).drop_duplicates(['vehicleCode', 'trip', 'date'])
print(f"Total trips: {len(trips):,}")
print()
print("Trip-level flags:")
print(f"  flag_negative_load:     {trips['flag_negative_load'].sum():,}")
print(f"  flag_capacity_exceeded: {trips['flag_capacity_exceeded'].sum():,}")

After rules: (1598277, 43)

Row-level flags:
  flag_gps_zero:            18,399
  flag_gps_outside_halland: 18,409

Total trips: 64,172

Trip-level flags:
  flag_negative_load:     24,943
  flag_capacity_exceeded: 1
